In [15]:
from datetime import date

import numpy as np
import pandas as pd
import yaml
from sqlalchemy import create_engine


# database connections 

In [16]:
with open('../config_fill.yml', 'r') as f:
    config = yaml.safe_load(f)
    config_co = config['CO_SA']
    config_etl = config['ETL_PRO']

url_co = (f"{config_co['drivername']}://{config_co['user']}:{config_co['password']}@{config_co['host']}:"
          f"{config_co['port']}/{config_co['dbname']}")
url_etl = (f"{config_etl['drivername']}://{config_etl['user']}:{config_etl['password']}@{config_etl['host']}:"
           f"{config_etl['port']}/{config_etl['dbname']}")
co_sa = create_engine(url_co)
etl_conn = create_engine(url_etl)

# Extract

In [17]:
dim_estado = pd.read_sql_table('mensajeria_estado', co_sa)
dim_estado.info()
dim_estado

<class 'pandas.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   id           6 non-null      int64
 1   nombre       6 non-null      str  
 2   descripcion  6 non-null      str  
dtypes: int64(1), str(2)
memory usage: 276.0 bytes


,id,nombre,descripcion
0,4,Recogido por mensajero,Recogido por mensajero
1,5,Entregado en destino,Entregado en destino
2,3,Con novedad,Tiene novedad
3,6,Terminado completo,Terminado completo
4,1,Iniciado,Creado por el usuario
5,2,Con mensajero Asignado,Con mensajero Asignado


# Transformations

El campo `descripcion` original duplica casi textualmente a `nombre` (ej. 'Recogido por mensajero' / 'Recogido por mensajero'), asi que no aporta y se descarta.

El `Orden_Secuencia` NO se puede tomar de ninguna columna fuente -- se calcula manualmente segun el flujo oficial del documento de diseno:
Iniciado(1) -> Con mensajero Asignado(2) -> Recogido por mensajero(3) -> Entregado en destino(4) -> Terminado completo(5).

'Con novedad' NO forma parte de esa secuencia principal (es un estado de excepcion, no un paso del flujo), por lo que se le asigna orden_secuencia = NaN/None.

In [18]:
dim_estado.drop(columns=['descripcion'], inplace=True)

In [19]:
# TODO: si tus companeros usan otros nombres de estado en su BD, ajusta este diccionario.
orden_secuencia_map = {
    'Iniciado': 1,
    'Con mensajero Asignado': 2,
    'Recogido por mensajero': 3,
    'Entregado en destino': 4,
    'Terminado completo': 5,
    'Con novedad': None,  # estado de excepcion, fuera del flujo principal
}

dim_estado['orden_secuencia'] = dim_estado['nombre'].map(orden_secuencia_map)

# Verificar que ningun estado quedo sin mapear por error de tipeo
sin_mapear = dim_estado[~dim_estado['nombre'].isin(orden_secuencia_map.keys())]
if not sin_mapear.empty:
    print('ATENCION: hay estados sin mapear, revisar nombres exactos:')
    print(sin_mapear)

dim_estado

,id,nombre,orden_secuencia
0,4,Recogido por mensajero,3.0
1,5,Entregado en destino,4.0
2,3,Con novedad,NaN
3,6,Terminado completo,5.0
4,1,Iniciado,1.0
5,2,Con mensajero Asignado,2.0


# Alinear nombres de columnas al diagrama
ID_Estado (PK) -> indice al cargar | Nombre_Estado | Orden_Secuencia

In [20]:
dim_estado.rename(columns={
    'nombre': 'nombre_estado',
}, inplace=True)
#dim_estado['saved'] = date.today
dim_estado

,id,nombre_estado,orden_secuencia
0,4,Recogido por mensajero,3.0
1,5,Entregado en destino,4.0
2,3,Con novedad,NaN
3,6,Terminado completo,5.0
4,1,Iniciado,1.0
5,2,Con mensajero Asignado,2.0


# load

In [21]:
# NOTA: cambiado de if_exists='replace' a 'append' porque la tabla ahora se crea por DDL
# (ver sqlscripts.yml) con su PK real. 'replace' borraria esa tabla y la reemplazaria sin
# restricciones. Antes de correr este notebook, asegurate que main.py ya ejecuto el DDL
# (o ejecuta el script de sqlscripts.yml manualmente si la tabla aun no existe).
dim_estado.to_sql('dim_estado', etl_conn, if_exists='append',index_label='key_dim_estado')

6